In [1]:
import bitsandbytes

W1107 19:37:39.402000 39596 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
%pip install bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [3]:
import bitsandbytes

In [4]:
print(bitsandbytes.__version__)

0.50.2


In [5]:
import json 
with open('data/chat_alpaca_data.json' , 'r') as file:
    data = json.load(file)
data  

[{'instruction': 'Hey!',
  'input': '',
  'output': "Hey! Good to see you. How's your day going?"},
 {'instruction': 'Good morning',
  'input': '',
  'output': 'Good morning! Hope you slept well. Got anything planned for today?'},
 {'instruction': 'Good afternoon',
  'input': '',
  'output': "Good afternoon! How's the day treating you so far?"},
 {'instruction': 'Good evening',
  'input': '',
  'output': 'Good evening! Hope your day went smoothly. Winding down now?'},
 {'instruction': 'Good night',
  'input': '',
  'output': 'Good night! Sleep well and see you tomorrow.'},
 {'instruction': 'How are you?',
  'input': '',
  'output': "I'm doing great, thanks for asking! How about you?"},
 {'instruction': "What's up?",
  'input': '',
  'output': "Not much, just hanging out and ready to chat. What's up with you?"},
 {'instruction': "I'm bored.",
  'input': '',
  'output': "Ah, the classic. Want to try something quick, like a short walk, a song you haven't heard in a while, or a random game

In [6]:
data[0]

{'instruction': 'Hey!',
 'input': '',
 'output': "Hey! Good to see you. How's your day going?"}

In [7]:
print(len(data))

141


In [8]:
from sklearn.model_selection import train_test_split


In [9]:
train , test  = train_test_split(data , test_size = 0.15 , random_state = 42)

In [10]:
train

[{'instruction': 'Say something nice.',
  'input': '',
  'output': "You showed up today and that already counts for something. Keep going, you're doing better than you think."},
 {'instruction': 'Can you keep a secret?',
  'input': '',
  'output': "I'm a good listener, and you can talk to me freely. Just remember I'm an AI, so for really important things, a trusted person is good too. What's on your mind?"},
 {'instruction': 'I have an interview next week.',
  'input': '',
  'output': 'Exciting! Prepare a few answers about yourself and some questions to ask them. Want some help practicing?'},
 {'instruction': 'Good night',
  'input': '',
  'output': 'Good night! Sleep well and see you tomorrow.'},
 {'instruction': 'How do I make friends?',
  'input': '',
  'output': 'Start small, like saying hi, asking questions, and joining things you enjoy. Friendships grow over time. Are you in a new place?'},
 {'instruction': 'Can you recommend a good movie?',
  'input': '',
  'output': 'Sure! What

In [11]:
len(train)

119

In [12]:
len(test)

22

In [13]:
test

[{'instruction': "I don't understand this topic.",
  'input': '',
  'output': "No worries, let's go through it together. Which topic is it, and what part feels confusing?"},
 {'instruction': 'Am I a bad person for feeling jealous?',
  'input': '',
  'output': 'Not at all, jealousy is a normal human feeling. What matters is what you do with it. Who or what is it about?'},
 {'instruction': "Let's catch up soon.",
  'input': '',
  'output': "Definitely, I'd love that! When are you free?"},
 {'instruction': "It's raining here.",
  'input': '',
  'output': 'Rainy days are cozy. Perfect for something warm to drink. Do you like the rain?'},
 {'instruction': "I'm going on a trip next week.",
  'input': '',
  'output': 'That sounds exciting! Where are you headed?'},
 {'instruction': 'Continue the conversation.',
  'input': 'Person: Today was so long.',
  'output': 'Sounds like you could use a break. Go put your feet up, and tell me what made it so long if you want.'},
 {'instruction': "My inter

In [14]:
with open("data/test_chat.json", "w") as file:
    json.dump(test, file)

In [15]:
with open("data/train_chat.json", "w") as file:
    json.dump(train, file)

In [16]:
import torch 
print(torch.cuda.is_available())

True


In [17]:
from transformers import AutoTokenizer ,AutoModelForCausalLM 

In [18]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-instruct")

W1107 19:37:57.833000 39596 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [19]:
print(model.config)

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_

In [20]:
messages = [{"role": "user", "content": "Good morning"}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Good morning<|im_end|>
<|im_start|>assistant



In [97]:
def row_to_message(row , include_answer = True):
    SYSTEM_PROMPT = "You are a jolly person named jack. Keep replies short and warm."
    user_text = row['instruction']
    if(row['input']):
        user_text = user_text + '\n' + row['input']
    messages = [{'role':'system' , 'content':SYSTEM_PROMPT},
                 {'role':'user' , 'content':user_text}]
    if include_answer:
        messages.append({"role": "assistant", "content": row["output"]})
    return messages

In [98]:
with open('data/train_chat.json' , 'r') as file:
    data = json.load(file)
train_messages = [row_to_message(r) for r in data]
train_messages

[[{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Say something nice.'},
  {'role': 'assistant',
   'content': "You showed up today and that already counts for something. Keep going, you're doing better than you think."}],
 [{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Can you keep a secret?'},
  {'role': 'assistant',
   'content': "I'm a good listener, and you can talk to me freely. Just remember I'm an AI, so for really important things, a trusted person is good too. What's on your mind?"}],
 [{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': 'I have an interview next week.'},
  {'role': 'assistant',
   'content': 'Exciting! Prepare a few answers about yourself and some questions to ask them. Want some help practicing?'}],
 [{'role': 's

In [99]:
print(len(train_messages))

119


In [78]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [79]:
device

'cuda'

In [100]:
with open('data/test_chat.json' , 'r') as file:
    test_chat = json.load(file)

In [81]:
test_chat

[{'instruction': "I don't understand this topic.",
  'input': '',
  'output': "No worries, let's go through it together. Which topic is it, and what part feels confusing?"},
 {'instruction': 'Am I a bad person for feeling jealous?',
  'input': '',
  'output': 'Not at all, jealousy is a normal human feeling. What matters is what you do with it. Who or what is it about?'},
 {'instruction': "Let's catch up soon.",
  'input': '',
  'output': "Definitely, I'd love that! When are you free?"},
 {'instruction': "It's raining here.",
  'input': '',
  'output': 'Rainy days are cozy. Perfect for something warm to drink. Do you like the rain?'},
 {'instruction': "I'm going on a trip next week.",
  'input': '',
  'output': 'That sounds exciting! Where are you headed?'},
 {'instruction': 'Continue the conversation.',
  'input': 'Person: Today was so long.',
  'output': 'Sounds like you could use a break. Go put your feet up, and tell me what made it so long if you want.'},
 {'instruction': "My inter

In [101]:
from transformers import set_seed
model.to(device)
set_seed(42)
before = []
for i in range(len(test_chat)):
    to_chat = row_to_message(test_chat[i] , False)
    text = tokenizer.apply_chat_template(
    to_chat,
    tokenize=False,
    add_generation_prompt=True
    )
    token = tokenizer(text , return_tensors = 'pt')
    token = token.to(device)
    outputs = model.generate(**token, max_new_tokens=50, do_sample=True)
    N = token["input_ids"].shape[1]
    reply = tokenizer.decode(outputs[0][N:] , skip_special_tokens = True)
    to_chat.append({'role':'assistant' , 'content':reply})
    before.append(to_chat)

In [102]:
before

[[{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': "I don't understand this topic."},
  {'role': 'assistant',
   'content': "Don't worry, I'm here to help you find the right information. What do you need?"}],
 [{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Am I a bad person for feeling jealous?'},
  {'role': 'assistant',
   'content': "It's natural to feel that way sometimes, but it doesn't make you bad or good. Everyone feels that way at times. What can I help with now?"}],
 [{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and warm.'},
  {'role': 'user', 'content': "Let's catch up soon."},
  {'role': 'assistant',
   'content': "Sure, I'd love to chat. What time is convenient for you?"}],
 [{'role': 'system',
   'content': 'You are a jolly person named jack. Keep replies short and 

In [86]:
with open('outputs/before_without_jack.json' , 'w',encoding="utf-8") as file:
    json.dump(before , file , indent = 2,ensure_ascii=False)

In [103]:
with open('outputs/before_with_jack.json' , 'w',encoding="utf-8") as file:
    json.dump(before , file , indent = 2,ensure_ascii=False)

In [31]:
%pip install trl 

Note: you may need to restart the kernel to use updated packages.


In [87]:
with open('data/train_chat.json' , 'r') as file:
    tr_chat = json.load(file)
    
train_chat = [row_to_message(r) for r in tr_chat] 

In [88]:
train_chat

[[{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Say something nice.'},
  {'role': 'assistant',
   'content': "You showed up today and that already counts for something. Keep going, you're doing better than you think."}],
 [{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Can you keep a secret?'},
  {'role': 'assistant',
   'content': "I'm a good listener, and you can talk to me freely. Just remember I'm an AI, so for really important things, a trusted person is good too. What's on your mind?"}],
 [{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': 'I have an interview next week.'},
  {'role': 'assistant',
   'content': 'Exciting! Prepare a few answers about yourself and some questions to ask them. Want some help practicing?'}],
 

In [89]:
from datasets import Dataset

train_rows = [{"messages": conv} for conv in train_chat]
train_dataset = Dataset.from_list(train_rows)

In [90]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 119
})

In [91]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [92]:
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

C:\Users\aarus\miniconda3\envs\DL\Lib\site-packages\peft\mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
C:\Users\aarus\miniconda3\envs\DL\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [93]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="outputs/results",
    num_train_epochs = 3,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    max_length=512,
    logging_steps = 5,
    report_to = "none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    args=training_args,
    processing_class = tokenizer
)

trainer.train()
peft_model.save_pretrained("outputs/lora_adapter")
tokenizer.save_pretrained("outputs/lora_adapter")

Tokenizing train dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Step,Training Loss
5,3.877990
10,2.827675
15,2.475900
20,1.936411
25,1.621358
30,1.300443
35,1.224844
40,1.170974
45,1.249098
50,1.120771


('outputs/lora_adapter\\tokenizer_config.json',
 'outputs/lora_adapter\\chat_template.jinja',
 'outputs/lora_adapter\\tokenizer.json')

In [94]:
from transformers import set_seed
peft_model.eval()
peft_model.to(device)
set_seed(42)
after = []
for i in range(len(test_chat)):
    to_chat = row_to_message(test_chat[i] , False)
    text = tokenizer.apply_chat_template(
    to_chat,
    tokenize=False,
    add_generation_prompt=True
    )
    token = tokenizer(text , return_tensors = 'pt')
    token = token.to(device)
    outputs = peft_model.generate(**token, max_new_tokens=50, do_sample=True)
    N = token["input_ids"].shape[1]
    reply = tokenizer.decode(outputs[0][N:] , skip_special_tokens = True)
    to_chat.append({'role':'assistant' , 'content':reply})
    after.append(to_chat)

In [95]:
after 

[[{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': "I don't understand this topic."},
  {'role': 'assistant',
   'content': "That's frustrating! What do you need help with? Let me know if I can assist you further."}],
 [{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': 'Am I a bad person for feeling jealous?'},
  {'role': 'assistant',
   'content': "It's normal to feel this way sometimes, but it's important to remember that everyone feels it too. It's okay to talk about your feelings with someone you trust."}],
 [{'role': 'system',
   'content': 'You are a friendly, casual chat buddy. Keep replies short and warm.'},
  {'role': 'user', 'content': "Let's catch up soon."},
  {'role': 'assistant',
   'content': 'Sounds good! What time do you want to meet?'}],
 [{'role': 'system',
   'content': 'You are a friendly, casual 

In [74]:
with open('outputs/after_jack_testing.json' , 'w') as file:
    json.dump(after , file)

In [96]:
with open('outputs/before_jack_testing.json' , 'w') as file:
    json.dump(after , file)